In [1]:
pip install selenium webdriver-manager


The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: ("Connection broken: OSError(28, 'No space left on device')", OSError(28, 'No space left on device'))



In [2]:
df -h

NameError: name 'df' is not defined

In [3]:
jupyter notebook --generate-config

SyntaxError: invalid syntax (1387249117.py, line 1)

In [4]:
pip install selenium webdriver-manager

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/9.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.4 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.4 MB 2.5 MB/s eta 0:00:04
   ----- ---------------------------------- 1.3/9.4 MB 2.3 MB/s eta 0:00:04
   ------- -------------------------------- 1.8/9.4 MB 2.4 MB/s eta 0:00:04
   ---------- ----------------------------- 2.4/9.4 MB 2.5 MB/s eta 0:00:03
   ------------ --------------------------- 2.9/9.4 MB 2.6 MB/s eta 0:00:03
   --------------- ------------------------ 3.7/9.4 MB 2.7 MB/s eta 0:00:03
   ----------------- ---------------------- 4.2/9.4 MB 2.7 MB/s eta 0:00:02
   ------------------ --------------------- 4.5/9.4 MB 2.5 MB/s eta 0:00:03
   ------------------ --------------------- 4.5/9.4 MB 2.5 MB/s eta 0:00:03
   --------------------- ------------------ 5.0/9.4 MB 2.2 MB/s eta 0:00:02
   ---------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [6]:

import os
import re
import time
import json
from datetime import datetime, timedelta
from urllib.parse import urljoin, urlparse

import requests
import pandas as pd
from bs4 import BeautifulSoup

try:
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options as ChromeOptions
    from selenium.webdriver.chrome.service import Service as ChromeService
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.common.exceptions import WebDriverException, TimeoutException
    SELENIUM_AVAILABLE = True
except ImportError:
    SELENIUM_AVAILABLE = False

try:
    from webdriver_manager.chrome import ChromeDriverManager
    WEBDRIVER_MANAGER_AVAILABLE = True
except ImportError:
    WEBDRIVER_MANAGER_AVAILABLE = False

try:
    from IPython.display import HTML, display as ipy_display
    IPY_AVAILABLE = True
except ImportError:
    IPY_AVAILABLE = False
    def ipy_display(*args, **kwargs):
        pass

# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================
COMPANY_NAME   = "HDFC Bank Limited"
SCREENER_SLUG  = "HDFCBANK"     # screener.in/company/HDFCBANK/
BSE_SCRIP_CODE = "500180"
NSE_SYMBOL     = "HDFCBANK"

LOOKBACK_DAYS   = 730              # how far back to pull BSE announcements
REQUEST_DELAY   = 1.0              # polite delay between downloads (seconds)
TIMEOUT         = 20

TODAY_STR      = datetime.now().strftime("%d-%b-%Y")
OUTPUT_ROOT    = f"{NSE_SYMBOL}_Disclosure_Documents_{TODAY_STR}"

SUBFOLDERS = {
    "annual_reports":        "01_Annual_Reports",
    "investor_presentations":"02_Investor_Presentations",
    "earnings_transcripts":  "03_Earnings_Call_Transcripts",
    "concall_recordings":    "04_Concall_Recordings",
    "credit_ratings":        "05_Credit_Rating_Reports",
    "quarterly_results":     "06_Quarterly_Result_Filings",
    "other_documents":       "07_Other_Disclosures",
}
for sub in SUBFOLDERS.values():
    os.makedirs(os.path.join(OUTPUT_ROOT, sub), exist_ok=True)

HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                    "(KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"),
    "Accept-Language": "en-US,en;q=0.9",
}
BSE_HEADERS = {**HEADERS, "Referer": "https://www.bseindia.com/", "Origin": "https://www.bseindia.com"}

session = requests.Session()
session.headers.update(HEADERS)

print(f"⚙️  Document fetch target: {COMPANY_NAME} | BSE: {BSE_SCRIP_CODE} | NSE: {NSE_SYMBOL}")
print(f"📁 Output root: ./{OUTPUT_ROOT}/")


def safe_filename(text, max_len=120):
    """Turn arbitrary link text/URL into a filesystem-safe filename."""
    text = re.sub(r'\s+', ' ', str(text)).strip()
    text = re.sub(r'[\\/:*?"<>|]', '-', text)
    return text[:max_len] if text else "document"


def categorize(text, url=""):
    """Keyword-based classification — robust to markup/layout changes since
    it only depends on link text and URL content, not exact HTML structure."""
    t = f"{text} {url}".lower()
    if "annual report" in t:
        return "annual_reports"
    if "credit rating" in t or "rating rationale" in t or re.search(r'\bcrisil\b|\bicra\b|\bcare\b|india ratings', t):
        return "credit_ratings"
    if "transcript" in t:
        return "earnings_transcripts"
    if "ppt" in t or "presentation" in t or "investor presentation" in t:
        return "investor_presentations"
    if "rec" in t.split() or "audio" in t or "concall" in t or "recording" in t:
        return "concall_recordings"
    if ("financial result" in t or "quarterly result" in t or "audited result" in t
            or "unaudited result" in t or re.search(r'\bresults?\b', t)):
        return "quarterly_results"
    if ("board meeting" in t) and re.search(r'\bquarter|\bfinancial year|\bq[1-4]\b', t):
        return "quarterly_results"
    return "other_documents"


def diagnose_response(resp):
    """
    Inspect a failed/suspicious Screener.in response and print a short,
    human-readable explanation of what likely happened — bot-detection
    challenge, redirect, rate limit, or genuine structural change — so a
    failure is self-explanatory instead of a bare status code.
    """
    try:
        soup = BeautifulSoup(resp.content, "html.parser")
        title_tag = soup.find("title")
        title = title_tag.get_text(strip=True) if title_tag else "(no <title> tag found)"

        challenge_markers = ["cloudflare", "captcha", "just a moment", "attention required",
                              "verify you are human", "access denied", "checking your browser"]
        body_lower = resp.text.lower()
        hits = [m for m in challenge_markers if m in body_lower]

        print("   🔍 Diagnostic:")
        print(f"      Final URL      : {resp.url}")
        print(f"      Page title     : {title}")
        print(f"      Redirects      : {len(resp.history)}")
        if hits:
            print(f"      Bot-challenge markers found: {hits}")
            print("      -> This looks like an anti-bot challenge page (e.g. Cloudflare), "
                  "not a genuine missing/blocked page. A plain requests+BeautifulSoup client "
                  "generally cannot pass this — it needs a real headless browser (Selenium/"
                  "Playwright) to execute the challenge's JavaScript.")
        else:
            print("      No known bot-challenge markers found in the response body.")
            print(f"      -> Contains '#documents' section: "
                  f"{'YES' if soup.find(id='documents') else 'NO'}")
            if resp.status_code == 404:
                print("      -> A clean 404 with no challenge markers suggests either the URL "
                      "pattern is wrong for this ticker, or Screener is returning a disguised "
                      "block page as a 404 rather than a real 'not found'.")
    except Exception as e:
        print(f"   (diagnostic inspection itself failed: {e})")


# ==============================================================================
# 2. SOURCE A — SCREENER.IN DOCUMENT HUB
# ==============================================================================
def fetch_screener_documents(slug):
    """
    Scrape the 'Documents' hub on a Screener.in company page, which aggregates
    official BSE/NSE PDF links for Annual Reports, Concalls (Transcript / PPT /
    REC), and Credit Ratings. Returns a list of dicts:
        {title, url, category, source}
    Never raises — returns [] with a printed warning (plus a short diagnostic)
    on any failure.
    """
    results = []
    try:
        # Warm up the session with a homepage visit first — some anti-bot
        # front-ends only serve a full page to a session that already has
        # cookies from a prior request, and will otherwise return 403/404 on
        # a cold direct hit to a deep company URL.
        try:
            session.get("https://www.screener.in/", timeout=TIMEOUT, headers={
                **HEADERS, "Referer": "https://www.google.com/"
            })
        except requests.exceptions.RequestException:
            pass  # non-fatal — proceed to the real request regardless

        page_headers = {**HEADERS, "Referer": "https://www.screener.in/"}
        url = f"https://www.screener.in/company/{slug}/consolidated/"
        resp = session.get(url, timeout=TIMEOUT, headers=page_headers)
        if resp.status_code != 200:
            url = f"https://www.screener.in/company/{slug}/"
            resp = session.get(url, timeout=TIMEOUT, headers=page_headers)

        if resp.status_code != 200:
            print(f"⚠️ Screener.in returned status {resp.status_code}; skipping this source.")
            diagnose_response(resp)
            return results

        soup = BeautifulSoup(resp.content, 'html.parser')

        doc_section = soup.find(id='documents') or soup.find(class_=re.compile(r'\bdocuments\b'))
        if not doc_section:
            print("⚠️ Screener.in 'Documents' section not found on page (layout may have changed).")
            diagnose_response(resp)
            return results

        for a in doc_section.find_all('a', href=True):
            href = a['href'].strip()
            if not href or href.startswith('#'):
                continue
            full_url = urljoin("https://www.screener.in", href)
            link_text = a.get_text(" ", strip=True) or os.path.basename(urlparse(full_url).path)

            # Try to recover a nearby heading (e.g. "Annual Reports", "Concalls",
            # "Credit Ratings") for extra context, but don't depend on it.
            heading = ""
            parent_block = a.find_parent(['div', 'li'])
            hop = parent_block
            for _ in range(4):
                if hop is None:
                    break
                head_el = hop.find_previous(['h1', 'h2', 'h3', 'div'],
                                             class_=re.compile(r'sub-title|title', re.I))
                if head_el and len(head_el.get_text(strip=True)) < 60:
                    heading = head_el.get_text(strip=True)
                    break
                hop = hop.find_parent(['div'])

            category = categorize(f"{heading} {link_text}", full_url)
            results.append({
                "title": link_text or heading or "Document",
                "url": full_url,
                "category": category,
                "source": "screener.in",
            })

        print(f"✅ Screener.in document hub: {len(results)} links discovered.")
    except requests.exceptions.RequestException as e:
        print(f"⚠️ Screener.in network error: {type(e).__name__}: {e}")
    except Exception as e:
        print(f"⚠️ Screener.in parsing error: {e}")
    return results


# ------------------------------------------------------------------------------
# 2b. SCREENER.IN VIA A REAL (HEADLESS) BROWSER — SELENIUM FALLBACK
# ------------------------------------------------------------------------------
# A plain `requests` call is a bare HTTP client: no JavaScript execution, no
# real browser fingerprint, no cookies from organic navigation. If Screener's
# edge is running any JS-based bot check, `requests` cannot pass it no matter
# how the headers are tuned. A real (headless) Chrome session executes the
# page exactly like a normal visitor's browser would, which is the standard
# fix for this class of blocking. This is used automatically as a fallback
# only if the plain-requests fetch above returns zero results.
#
# Requirements on YOUR machine (not this sandbox):
#   pip install selenium webdriver-manager
#   A real Chrome or Chromium browser installed (webdriver-manager only
#   manages the matching *driver*, not the browser itself).
def build_headless_chrome_driver():
    """Construct a headless Chrome driver with reduced automation fingerprints.
    Returns None (never raises) if selenium/webdriver-manager/Chrome itself
    is unavailable, so callers can fall back gracefully."""
    if not SELENIUM_AVAILABLE:
        print("   ℹ️ selenium not installed. Run: pip install selenium webdriver-manager")
        return None

    options = ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1366,900")
    options.add_argument(f"--user-agent={HEADERS['User-Agent']}")
    # Reduce the most common automation fingerprints that basic bot checks key on.
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)

    try:
        if WEBDRIVER_MANAGER_AVAILABLE:
            service = ChromeService(ChromeDriverManager().install())
            driver = webdriver.Chrome(service=service, options=options)
        else:
            # Assumes 'chromedriver' is already on PATH and matches the
            # installed Chrome version.
            driver = webdriver.Chrome(options=options)

        # Patch the navigator.webdriver flag that Selenium sets by default —
        # one of the most commonly checked automation signals.
        try:
            driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
                "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
            })
        except Exception:
            pass  # non-fatal — proceed without the patch if CDP isn't available

        return driver

    except WebDriverException as e:
        print(f"   ⚠️ Could not start Chrome/chromedriver: {e}")
        print("      Make sure Google Chrome (or Chromium) is installed on this machine.")
        return None
    except Exception as e:
        print(f"   ⚠️ Unexpected error starting the browser: {e}")
        return None


def fetch_screener_documents_selenium(slug, page_load_timeout=25):
    """
    Same goal as fetch_screener_documents() — pull Annual Report / Concall /
    Credit Rating PDF links from Screener's Documents hub — but rendered
    through a real headless Chrome session instead of a bare HTTP request.
    Never raises. Returns [] with a printed reason on any failure, and always
    quits the browser in a `finally` block so a failure never leaves a
    zombie Chrome process behind.
    """
    results = []
    driver = None
    try:
        print("   🌐 Launching headless Chrome for Screener.in (Selenium fallback)...")
        driver = build_headless_chrome_driver()
        if driver is None:
            return results

        driver.set_page_load_timeout(page_load_timeout)

        url = f"https://www.screener.in/company/{slug}/consolidated/"
        try:
            driver.get(url)
        except TimeoutException:
            print("   ⚠️ Page load timed out; trying the non-consolidated URL...")
            url = f"https://www.screener.in/company/{slug}/"
            driver.get(url)

        # Give the page a real chance to render / pass any JS challenge.
        try:
            WebDriverWait(driver, page_load_timeout).until(
                lambda d: d.execute_script("return document.readyState") == "complete"
            )
        except TimeoutException:
            pass  # proceed with whatever rendered so far

        # If a #documents section never appears, wait a little longer in case
        # it's lazy-loaded, then give up gracefully rather than hanging.
        try:
            WebDriverWait(driver, 8).until(
                EC.presence_of_element_located((By.ID, "documents"))
            )
        except TimeoutException:
            pass

        page_title = driver.title or ""
        page_source = driver.page_source

        challenge_markers = ["just a moment", "attention required", "verify you are human",
                              "checking your browser", "captcha"]
        if any(m in page_source.lower() or m in page_title.lower() for m in challenge_markers):
            print(f"   ⚠️ Still hit a bot-challenge page even via headless browser (title: '{page_title}'). "
                  "This means the site requires actual CAPTCHA-solving or is detecting headless mode "
                  "specifically — beyond what a script can reliably automate.")
            return results

        soup = BeautifulSoup(page_source, "html.parser")
        doc_section = soup.find(id="documents") or soup.find(class_=re.compile(r"\bdocuments\b"))
        if not doc_section:
            print(f"   ⚠️ Rendered page (title: '{page_title}') still has no '#documents' section — "
                  "Screener's layout for this page may have changed.")
            return results

        for a in doc_section.find_all("a", href=True):
            href = a["href"].strip()
            if not href or href.startswith("#"):
                continue
            full_url = urljoin("https://www.screener.in", href)
            link_text = a.get_text(" ", strip=True) or os.path.basename(urlparse(full_url).path)
            category = categorize(link_text, full_url)
            results.append({
                "title": link_text or "Document",
                "url": full_url,
                "category": category,
                "source": "screener.in (selenium)",
            })

        print(f"   ✅ Selenium fetch: {len(results)} links discovered.")

    except Exception as e:
        print(f"   ⚠️ Selenium fetch failed unexpectedly: {e}")
    finally:
        if driver is not None:
            try:
                driver.quit()
            except Exception:
                pass
    return results


# ==============================================================================
# 3. SOURCE B — BSE CORPORATE ANNOUNCEMENTS API (best-effort)
# ==============================================================================
# BSE category codes used by the public announcements search widget. These are
# community-reverse-engineered and BSE updates them occasionally — the fetch
# is fully wrapped so a schema change degrades to "0 results found" instead
# of a crash.
BSE_CATEGORY_MAP = {
    "Annual Report":                 "annual_reports",
    "Financial Results":             "quarterly_results",
    "Credit Rating":                 "credit_ratings",
    "Investor Presentation":         "investor_presentations",
    "Earnings Call / Con-call":      "earnings_transcripts",
}
# NOTE: "Company Update" is deliberately NOT in this map. BSE tags almost
# every Regulation 30 disclosure (investor presentations, transcripts, press
# releases, ESOP allotments, etc.) with the same generic CATEGORYNAME of
# "Company Update" — mapping it to any single bucket would short-circuit the
# more specific keyword classifier below and dump everything into one folder.


def resolve_category(subject, url, category_raw):
    """
    Keyword-based classification on the SUBJECT LINE takes priority, since
    BSE's own CATEGORYNAME field is too coarse (almost everything is tagged
    "Company Update" regardless of actual content). The BSE category label is
    only used as a fallback when the subject-line keywords are inconclusive.
    """
    keyword_guess = categorize(subject, url)
    if keyword_guess != "other_documents":
        return keyword_guess
    bse_guess = BSE_CATEGORY_MAP.get((category_raw or "").strip())
    return bse_guess or "other_documents"


def fetch_bse_announcements(scrip_code, lookback_days=LOOKBACK_DAYS):
    """
    Best-effort pull of corporate-announcement metadata from BSE's public JSON
    endpoint (as used by the bseindia.com website's own announcement widget).
    Returns a list of dicts: {title, url, category, source, date}.
    Never raises — returns [] with a printed warning on any failure, including
    endpoint changes, non-JSON responses, blocked/geofenced IPs, or timeouts.
    """
    results = []
    try:
        to_date = datetime.now()
        from_date = to_date - timedelta(days=lookback_days)

        endpoint = "https://api.bseindia.com/BseIndiaAPI/api/AnnSubCategoryGetData/w"
        params = {
            "pageno": 1,
            "strCat": -1,
            "strPrevDate": from_date.strftime("%Y%m%d"),
            "strScrip": scrip_code,
            "strSearch": "P",
            "strToDate": to_date.strftime("%Y%m%d"),
            "strType": "C",
            "subcategory": -1,
        }
        resp = session.get(endpoint, params=params, headers=BSE_HEADERS, timeout=TIMEOUT)

        if resp.status_code != 200:
            print(f"⚠️ BSE announcements API returned status {resp.status_code}; skipping this source.")
            return results

        try:
            data = resp.json()
        except json.JSONDecodeError:
            print("⚠️ BSE announcements API did not return valid JSON (endpoint may have changed); skipping.")
            return results

        rows = data.get("Table", []) if isinstance(data, dict) else []
        if not rows:
            print("ℹ️ BSE announcements API returned 0 rows for the lookback window.")
            return results

        for row in rows:
            try:
                subject   = row.get("NEWSSUB") or row.get("HEADLINE") or "BSE Announcement"
                news_dt   = row.get("NEWS_DT") or row.get("DissemDT") or ""
                attach    = row.get("ATTACHMENTNAME") or ""
                category_raw = row.get("CATEGORYNAME") or row.get("CATEGORY") or ""

                if not attach:
                    continue  # nothing downloadable for this row

                # BSE serves recent filings from AttachLive/ and moves older
                # ones to AttachHis/ after some weeks. Provide both as
                # candidate URLs — the downloader tries them in order and
                # falls back automatically, which fixes the "HTTP 404 on
                # anything more than a few weeks old" failure mode.
                live_url = f"https://www.bseindia.com/xml-data/corpfiling/AttachLive/{attach}"
                hist_url = f"https://www.bseindia.com/xml-data/corpfiling/AttachHis/{attach}"
                category = resolve_category(subject, live_url, category_raw)

                results.append({
                    "title": subject,
                    "url": live_url,
                    "fallback_urls": [hist_url],
                    "category": category,
                    "source": "bseindia.com",
                    "date": news_dt,
                })
            except Exception:
                continue  # skip malformed row, keep processing the rest

        print(f"✅ BSE announcements API: {len(results)} downloadable filings discovered.")
    except requests.exceptions.RequestException as e:
        print(f"⚠️ BSE API network error: {type(e).__name__}: {e}")
    except Exception as e:
        print(f"⚠️ BSE API unexpected error: {e}")
    return results


# ==============================================================================
# 4. DOWNLOAD ENGINE
# ==============================================================================
def download_document(doc, dest_root, retries=2):
    """
    Download a single document dict {title, url, category, source, ...}.
    Returns an updated dict with 'local_path' and 'status' fields.
    Verifies the response actually looks like a PDF (magic bytes) when the
    URL claims to be one; otherwise saves whatever content-type was served
    (e.g. an HTML landing page is saved as .html rather than mislabeled .pdf).
    Never raises.
    """
    doc = dict(doc)  # don't mutate caller's dict
    category = doc.get("category", "other_documents")
    subfolder = SUBFOLDERS.get(category, SUBFOLDERS["other_documents"])
    folder_path = os.path.join(dest_root, subfolder)

    try:
        os.makedirs(folder_path, exist_ok=True)
    except Exception as e:
        doc["local_path"] = None
        doc["status"] = f"failed (could not create destination folder: {e})"
        return doc

    base_name = safe_filename(doc.get("title", "document"))
    ext_guess = os.path.splitext(urlparse(doc["url"]).path)[1] or ".pdf"
    filename = f"{base_name}{ext_guess if ext_guess.lower() in ('.pdf', '.html', '.htm') else '.pdf'}"
    dest_path = os.path.join(folder_path, filename)

    # Avoid re-downloading if already present
    if os.path.exists(dest_path) and os.path.getsize(dest_path) > 1024:
        doc["local_path"] = dest_path
        doc["status"] = "already_exists"
        return doc

    candidate_urls = [doc["url"]] + list(doc.get("fallback_urls", []))
    last_error = None

    for url_index, candidate_url in enumerate(candidate_urls):
        for attempt in range(1, retries + 1):
            try:
                resp = session.get(candidate_url, headers=HEADERS, timeout=TIMEOUT, stream=True)
                if resp.status_code != 200:
                    last_error = f"HTTP {resp.status_code} ({candidate_url})"
                    time.sleep(REQUEST_DELAY)
                    continue

                content = resp.content
                if not content or len(content) < 512:
                    last_error = f"Empty or too-small response ({candidate_url})"
                    time.sleep(REQUEST_DELAY)
                    continue

                # If it doesn't look like a PDF, re-label the extension honestly
                # rather than silently saving mislabeled content.
                if not content.startswith(b'%PDF') and filename.lower().endswith('.pdf'):
                    content_type = resp.headers.get('Content-Type', '')
                    if 'html' in content_type.lower():
                        dest_path = dest_path[:-4] + ".html"
                        doc["status"] = "saved_as_html (not a direct PDF link)"
                    else:
                        doc["status"] = "saved (non-PDF content-type)"
                else:
                    doc["status"] = "downloaded" if url_index == 0 else "downloaded (via fallback archive URL)"

                with open(dest_path, "wb") as f:
                    f.write(content)

                doc["url"] = candidate_url  # record the URL that actually worked
                doc["local_path"] = dest_path
                return doc

            except requests.exceptions.RequestException as e:
                last_error = f"{type(e).__name__}: {e} ({candidate_url})"
                time.sleep(REQUEST_DELAY)
            except Exception as e:
                last_error = f"Unexpected error: {e} ({candidate_url})"
                break

    doc["local_path"] = None
    doc["status"] = f"failed ({last_error})"
    return doc


def download_all(doc_list, dest_root, delay=REQUEST_DELAY):
    """Download every document in doc_list, one at a time with a polite delay,
    isolating failures so the batch always completes."""
    downloaded = []
    for i, doc in enumerate(doc_list, 1):
        try:
            result = download_document(doc, dest_root)
        except Exception as e:
            result = {**doc, "local_path": None, "status": f"failed (unhandled: {e})"}
        downloaded.append(result)
        tag = "✅" if result["status"] in ("downloaded", "already_exists") else "⚠️"
        print(f"{tag} [{i}/{len(doc_list)}] {result.get('category')}: "
              f"{result.get('title', '')[:60]} -> {result['status']}")
        time.sleep(delay)
    return downloaded


# ==============================================================================
# 5. RUN THE PIPELINE
# ==============================================================================
print("\n🔎 Discovering documents (Screener.in + BSE)...")
screener_docs = fetch_screener_documents(SCREENER_SLUG)

if not screener_docs:
    if SELENIUM_AVAILABLE:
        print("ℹ️ Plain-requests fetch found nothing from Screener.in — trying the headless-browser fallback...")
        screener_docs = fetch_screener_documents_selenium(SCREENER_SLUG)
    else:
        print("ℹ️ Plain-requests fetch found nothing from Screener.in. A headless-browser fallback is "
              "available but not installed — run: pip install selenium webdriver-manager")

bse_docs = fetch_bse_announcements(BSE_SCRIP_CODE)

all_docs = screener_docs + bse_docs

# De-duplicate by URL (same filing can surface via both sources)
seen_urls = set()
deduped_docs = []
for d in all_docs:
    if d["url"] not in seen_urls:
        seen_urls.add(d["url"])
        deduped_docs.append(d)

print(f"\n📋 Total unique documents discovered: {len(deduped_docs)} "
      f"(Screener.in: {len(screener_docs)}, BSE: {len(bse_docs)})")

if not deduped_docs:
    print("⚠️ No documents discovered from either source. This commonly happens when:")
    print("   - Running from a blocked/geofenced/cloud IP (common for Screener & BSE)")
    print("   - Screener's page layout has changed since this script was written")
    print("   - BSE's announcement API schema has changed")
    print("   No files will be downloaded, but an (empty) manifest will still be produced below.")

print("\n⬇️  Downloading documents...")
downloaded_docs = download_all(deduped_docs, OUTPUT_ROOT) if deduped_docs else []

# ==============================================================================
# 6. MANIFEST (always produced, even if empty)
# ==============================================================================
manifest_cols = ["title", "category", "source", "date", "url", "local_path", "status"]
if downloaded_docs:
    df_manifest = pd.DataFrame(downloaded_docs)
    for c in manifest_cols:
        if c not in df_manifest.columns:
            df_manifest[c] = ""
    df_manifest = df_manifest[manifest_cols]
else:
    df_manifest = pd.DataFrame(columns=manifest_cols)

manifest_csv_path = os.path.join(OUTPUT_ROOT, "document_manifest.csv")
try:
    df_manifest.to_csv(manifest_csv_path, index=False)
    print(f"\n✅ Manifest saved: {manifest_csv_path}")
except Exception as e:
    print(f"⚠️ Could not write manifest CSV: {e}")

# ==============================================================================
# 7. SUMMARY DISPLAY
# ==============================================================================
print("\n" + "=" * 75)
print(f"🎉 DOCUMENT FETCH COMPLETE FOR: {COMPANY_NAME}")
print("=" * 75)

success_count = sum(1 for d in downloaded_docs if d["status"] in ("downloaded", "already_exists"))
print(f"📥 Successfully retrieved: {success_count} / {len(downloaded_docs)} documents")

if not df_manifest.empty:
    print("\n📊 By category:")
    print(df_manifest.groupby("category")["status"].count().to_string())
    ipy_display(df_manifest)

summary_html = f"""
<div style="border:2px solid #102C57; border-radius:8px; padding:15px; background-color:#F8FAFC; margin-top:15px; font-family:sans-serif;">
    <h3 style="color:#102C57; margin-top:0;">📁 Document Bundle: <i>./{OUTPUT_ROOT}/</i></h3>
    <ul>
        <li><b>Annual Reports:</b> {SUBFOLDERS['annual_reports']}/</li>
        <li><b>Investor Presentations:</b> {SUBFOLDERS['investor_presentations']}/</li>
        <li><b>Earnings Call Transcripts:</b> {SUBFOLDERS['earnings_transcripts']}/</li>
        <li><b>Concall Recordings:</b> {SUBFOLDERS['concall_recordings']}/</li>
        <li><b>Credit Rating Reports:</b> {SUBFOLDERS['credit_ratings']}/</li>
        <li><b>Quarterly Result Filings:</b> {SUBFOLDERS['quarterly_results']}/</li>
        <li><b>Manifest:</b> document_manifest.csv</li>
    </ul>
    <p style="color:#555; font-size:0.85em;">
        Retrieved {success_count} of {len(downloaded_docs)} discovered documents.
        Run again later to pick up newly filed disclosures (already-downloaded files are skipped).
    </p>
</div>
"""
if IPY_AVAILABLE:
    ipy_display(HTML(summary_html))
else:
    print(f"\nDocument bundle ready in ./{OUTPUT_ROOT}/")

⚙️  Document fetch target: HDFC Bank Limited | BSE: 500180 | NSE: HDFCBANK
📁 Output root: ./HDFCBANK_Disclosure_Documents_31-Jul-2026/

🔎 Discovering documents (Screener.in + BSE)...
✅ Screener.in document hub: 84 links discovered.
✅ BSE announcements API: 49 downloadable filings discovered.

📋 Total unique documents discovered: 132 (Screener.in: 84, BSE: 49)

⬇️  Downloading documents...
⚠️ [1/132] other_documents: All -> saved_as_html (not a direct PDF link)
✅ [2/132] other_documents: HDFC Bank Limited Announces Conclusion Of Internal Review Pe -> downloaded
✅ [3/132] other_documents: Intimation Under Regulation 30 Of The SEBI (LODR) Regulation -> downloaded
✅ [4/132] earnings_transcripts: Announcement under Regulation 30 (LODR)-Analyst / Investor M -> downloaded
✅ [5/132] other_documents: Announcement under Regulation 30 (LODR)-Allotment of ESOP /  -> downloaded
✅ [6/132] concall_recordings: Announcement under Regulation 30 (LODR)-Analyst / Investor M -> downloaded
✅ [7/132] other_d

,title,category,source,date,url,local_path,status
0,All,other_documents,screener.in,NaN,https://www.bseindia.com/stock-share-price/hdf...,HDFCBANK_Disclosure_Documents_31-Jul-2026\07_O...,saved_as_html (not a direct PDF link)
1,HDFC Bank Limited Announces Conclusion Of Inte...,other_documents,screener.in,NaN,https://www.bseindia.com/stockinfo/AnnPdfOpen....,HDFCBANK_Disclosure_Documents_31-Jul-2026\07_O...,downloaded
2,Intimation Under Regulation 30 Of The SEBI (LO...,other_documents,screener.in,NaN,https://www.bseindia.com/stockinfo/AnnPdfOpen....,HDFCBANK_Disclosure_Documents_31-Jul-2026\07_O...,downloaded
3,Announcement under Regulation 30 (LODR)-Analys...,earnings_transcripts,screener.in,NaN,https://www.bseindia.com/stockinfo/AnnPdfOpen....,HDFCBANK_Disclosure_Documents_31-Jul-2026\03_E...,downloaded
4,Announcement under Regulation 30 (LODR)-Allotm...,other_documents,screener.in,NaN,https://www.bseindia.com/stockinfo/AnnPdfOpen....,HDFCBANK_Disclosure_Documents_31-Jul-2026\07_O...,downloaded
...,...,...,...,...,...,...,...
127,Financial Results Including The Audited Standa...,quarterly_results,bseindia.com,2026-04-18T15:03:17.46,https://www.bseindia.com/xml-data/corpfiling/A...,HDFCBANK_Disclosure_Documents_31-Jul-2026\06_Q...,saved_as_html (not a direct PDF link)
128,Corporate Action-Board approves Dividend,other_documents,bseindia.com,2026-04-18T14:55:49.767,https://www.bseindia.com/xml-data/corpfiling/A...,HDFCBANK_Disclosure_Documents_31-Jul-2026\07_O...,downloaded (via fallback archive URL)
129,Board Meeting Outcome for Outcome Of The Board...,other_documents,bseindia.com,2026-04-18T14:45:54.733,https://www.bseindia.com/xml-data/corpfiling/A...,HDFCBANK_Disclosure_Documents_31-Jul-2026\07_O...,downloaded (via fallback archive URL)
130,Investment In The Equity Shares Of HDFC Life I...,other_documents,bseindia.com,2026-04-16T16:40:14.603,https://www.bseindia.com/xml-data/corpfiling/A...,HDFCBANK_Disclosure_Documents_31-Jul-2026\07_O...,downloaded (via fallback archive URL)


In [1]:
!git init

Reinitialized existing Git repository in C:/Users/ASUS/.git/


In [2]:
!git status

On branch main
Changes not staged for commit:
  (use "git add/rm <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	deleted:    README.md

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.VirtualBox/
	.android/
	.angular-config.json
	.antigravity-ide/
	.antigravity/
	.bash_history
	.cache/
	.config/
	.copilot/
	.cursor/
	.dbclient/
	.docker/
	.expo/
	.gemini/
	.ghcp-appmod/
	.gitconfig
	.idea/
	.idlerc/
	.ipynb_checkpoints/
	.ipython/
	.jdks/
	.jupyter/
	.matplotlib/
	.ssh/
	.viminfo
	.vscode-shared/
	.vscode/
	.wdm/
	AppData/
	Contacts/
	Downloads/
	Favorites/
	HDFCBANK_Disclosure_Documents_31-Jul-2026/
	IdeaProjects/
	Language-Learning-App/
	Links/
	LiveOpt/
	Music/
	NTUSER.DAT
	NTUSER.DAT{b6a461d4-bb8e-11ef-b6a1-9aeb161dd9e6}.TM.blf
	NTUSER.DAT{b6a461d4-bb8e-11ef-b6a1-9aeb161dd9e6}.TMContainer00000000000000000001.regtrans-ms
	NTUSER.DAT{b6a461d4-bb8e-11ef-b6a1-9aeb161dd9e6}.TMContai

In [3]:
!git commit -m "Initial commit: Financial intelligence harvester pipeline"


On branch main
Changes not staged for commit:
  (use "git add/rm <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	deleted:    README.md

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.VirtualBox/
	.android/
	.angular-config.json
	.antigravity-ide/
	.antigravity/
	.bash_history
	.cache/
	.config/
	.copilot/
	.cursor/
	.dbclient/
	.docker/
	.expo/
	.gemini/
	.ghcp-appmod/
	.gitconfig
	.idea/
	.idlerc/
	.ipynb_checkpoints/
	.ipython/
	.jdks/
	.jupyter/
	.matplotlib/
	.ssh/
	.viminfo
	.vscode-shared/
	.vscode/
	.wdm/
	AppData/
	Contacts/
	Downloads/
	Favorites/
	HDFCBANK_Disclosure_Documents_31-Jul-2026/
	IdeaProjects/
	Language-Learning-App/
	Links/
	LiveOpt/
	Music/
	NTUSER.DAT
	NTUSER.DAT{b6a461d4-bb8e-11ef-b6a1-9aeb161dd9e6}.TM.blf
	NTUSER.DAT{b6a461d4-bb8e-11ef-b6a1-9aeb161dd9e6}.TMContainer00000000000000000001.regtrans-ms
	NTUSER.DAT{b6a461d4-bb8e-11ef-b6a1-9aeb161dd9e6}.TMContai

In [4]:
!rmdir /s /q .git

In [5]:
import os
print(os.getcwd())

C:\Users\ASUS


In [6]:
import os

# 1. Create a dedicated project folder name
PROJECT_FOLDER = "financial-intelligence-harvester"
os.makedirs(PROJECT_FOLDER, exist_ok=True)

# 2. Change the working directory to that folder
os.chdir(PROJECT_FOLDER)

print(f"📁 Active Directory changed to: {os.getcwd()}")

📁 Active Directory changed to: C:\Users\ASUS\financial-intelligence-harvester


In [7]:
!git init

Initialized empty Git repository in C:/Users/ASUS/financial-intelligence-harvester/.git/


In [8]:
!git status

On branch master

No commits yet

nothing to commit (create/copy files and use "git add" to track)


In [9]:
# 1. Create .gitignore (Blocks heavy files)
gitignore_content = """*_Institutional_Report_*
*_Disclosure_Documents_*
*.pdf
*.xlsx
*.csv
*.png
.ipynb_checkpoints/
__pycache__/
*.pyc
.DS_Store
Thumbs.db
"""
with open(".gitignore", "w") as f:
    f.write(gitignore_content)

# 2. Create requirements.txt
requirements_content = """requests>=2.31.0
beautifulsoup4>=4.12.0
pandas>=2.0.0
numpy>=1.24.0
matplotlib>=3.7.0
fpdf2>=2.7.0
openpyxl>=3.1.0
ipython>=8.0.0
"""
with open("requirements.txt", "w") as f:
    f.write(requirements_content)

# 3. Create README.md
readme_content = """# 📊 Indian Equities Financial Intelligence Harvester

An automated, resilient Python-based financial data pipeline and report generator for Indian equities (NSE/BSE). This tool fetches live quarterly and annual financial statements, key valuation metrics, and disclosure documents, automatically generating publication-ready PDF reports, Excel workbooks, and visualization charts.

## 🛠️ Features
- Live scraping from financial portals (Screener.in)
- Auto-generated multi-page institutional PDF reports
- Master Excel workbook exports (`.xlsx`)
- Visual trend analysis charts
"""
with open("README.md", "w") as f:
    f.write(readme_content)

print("✅ Project files (.gitignore, requirements.txt, README.md) created successfully!")

UnicodeEncodeError: 'charmap' codec can't encode character '\U0001f4ca' in position 2: character maps to <undefined>

In [10]:
# 1. Create .gitignore (Blocks heavy files)
gitignore_content = """*_Institutional_Report_*
*_Disclosure_Documents_*
*.pdf
*.xlsx
*.csv
*.png
.ipynb_checkpoints/
__pycache__/
*.pyc
.DS_Store
Thumbs.db
"""
with open(".gitignore", "w", encoding="utf-8") as f:
    f.write(gitignore_content)

# 2. Create requirements.txt
requirements_content = """requests>=2.31.0
beautifulsoup4>=4.12.0
pandas>=2.0.0
numpy>=1.24.0
matplotlib>=3.7.0
fpdf2>=2.7.0
openpyxl>=3.1.0
ipython>=8.0.0
"""
with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements_content)

# 3. Create README.md
readme_content = """# 📊 Indian Equities Financial Intelligence Harvester

An automated, resilient Python-based financial data pipeline and report generator for Indian equities (NSE/BSE). This tool fetches live quarterly and annual financial statements, key valuation metrics, and disclosure documents, automatically generating publication-ready PDF reports, Excel workbooks, and visualization charts.

## 🛠️ Features
- Live scraping from financial portals (Screener.in)
- Auto-generated multi-page institutional PDF reports
- Master Excel workbook exports (`.xlsx`)
- Visual trend analysis charts
"""
with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

print("Project files (.gitignore, requirements.txt, README.md) created successfully!")

Project files (.gitignore, requirements.txt, README.md) created successfully!


In [11]:
!git status

On branch master

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.gitignore
	README.md
	requirements.txt

nothing added to commit but untracked files present (use "git add" to track)


In [12]:
!git status


On branch master

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.gitignore
	README.md
	requirements.txt

nothing added to commit but untracked files present (use "git add" to track)


In [13]:
!git add .

In [14]:
!git commit -m "Initial commit: Indian Equities Financial Intelligence Harvester"

[master (root-commit) 0f428ad] Initial commit: Indian Equities Financial Intelligence Harvester
 3 files changed, 28 insertions(+)
 create mode 100644 .gitignore
 create mode 100644 README.md
 create mode 100644 requirements.txt


In [15]:
!git add harvester_pipeline.ipynb
!git commit -m "Add core financial harvester notebook"

fatal: pathspec 'harvester_pipeline.ipynb' did not match any files


On branch master
nothing to commit, working tree clean


In [16]:
!git add harvester_pipeline.ipynb
!git commit -m "Add core financial harvester notebook"

fatal: pathspec 'harvester_pipeline.ipynb' did not match any files


On branch master
nothing to commit, working tree clean


In [17]:
!git add harvester_pipeline.ipynb
!git commit -m "Add core financial harvester notebook"

fatal: pathspec 'harvester_pipeline.ipynb' did not match any files


On branch master
nothing to commit, working tree clean


In [18]:
!git add .
!git commit -m "Add core financial harvester notebook"

On branch master
nothing to commit, working tree clean


In [19]:
!git branch -M main
!git remote add origin https://github.com/ArunkumarAK213/financial-intelligence-harvester.git
!git push -u origin main

remote: Repository not found.
fatal: repository 'https://github.com/ArunkumarAK213/financial-intelligence-harvester.git/' not found


In [20]:
!git remote remove origin
!git remote add origin https://github.com/ArunkumarAK213/financial-intelligence-Report harvester.git

usage: git remote add [<options>] <name> <url>

    -f, --[no-]fetch      fetch the remote branches
    --[no-]tags           import all tags and associated objects when fetching
                          or do not fetch any tag at all (--no-tags)
    -t, --[no-]track <branch>
                          branch(es) to track
    -m, --[no-]master <branch>
                          master branch
    --[no-]mirror[=(push|fetch)]
                          set up remote as a mirror to push to or fetch from



In [21]:
# 1. Clear any broken remote setting
!git remote remove origin

# 2. Re-add the URL without any spaces
!git remote add origin https://github.com/ArunkumarAK213/financial-intelligence-harvester.git

# 3. Push to main
!git push -u origin main

error: No such remote: 'origin'
remote: Repository not found.
fatal: repository 'https://github.com/ArunkumarAK213/financial-intelligence-harvester.git/' not found


In [22]:
# 1. Reset the remote origin
!git remote remove origin

# 2. Add your exact repository URL
!git remote add origin https://github.com/ArunkumarAK213/financial-intelligence-Report-harvester.git

# 3. Push your code to main
!git push -u origin main

branch 'main' set up to track 'origin/main'.


To https://github.com/ArunkumarAK213/financial-intelligence-Report-harvester.git
 * [new branch]      main -> main


In [23]:
!git add .
!git commit -m "Add core financial harvester notebook"
!git push origin main

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


Everything up-to-date
